In [2]:
# Transformers undet the hood: Attention is all you need (https://arxiv.org/pdf/1706.03762)

# GPT: Generativaly Pre-trained Transformers

In [3]:
# Dataset: Tiny Shakesphere

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print("char length: ",len(text))

char length:  1115393


In [7]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [9]:
chars = sorted(list(set(text))) # sorted list of all the char that occures
vocab_size = len(chars)  # size of vocabulary
print(''.join(chars))
print(vocab_size)

# 0 is '\n' and 1 is ''


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [19]:
# tokenization of input text (vocab to numbers)
# char level lang model
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of int, output a string

print(encode("hi there"))
print(decode(encode("hi there")))

# others:  openAI-tiktokens, google-Sentencebit

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [20]:
# Encoding entire dataset

import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [21]:
# data split (tr: 90%, test:10%)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [23]:
# how we train a transformer:
    # To compute heavy to do it all at once, so we sample random chunks and train iterativily on them
    # max len of chunks ( block_size )

In [25]:
# how transformers work? unlike RNNs and CNNs, transformers train the block_size in parallel not iterative 
# So, a block size of 8 means, training 8 threads starting with a pair (1 prev predicts next) then (2 prev predicts 
# next) ... total 8

block_size = 8
train_data[:block_size+1] # why +1, see above

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [28]:
# This is done so that transformers learns connections from as little as 1 input to block_size inputs
# So that sampling can be done from the begining without any reference

# this is the Time dimension: what comes next (pridicting future char)
x = train_data[:block_size] #(0, 7)
y = train_data[1:block_size + 1] #(1, 8)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [29]:
# next dimension Batch dimension: